# UZK: GRU без автоэнкодера

Запускайте из каталога проекта или ноутбука с ядром `.venv` после `uv sync`.
Конфиг: `configs/uzk/gru.yaml`.

Пайплайн исключает заданные интервалы, фильтрует y, масштабирует признаки, выбирает восемь
признаков из конфига, выравнивает будущую цель и формирует окна для GRU.
Скейлеры и GRU обучаются в этом запуске. Test вычисляется после обучения.


In [ ]:
import os
from pathlib import Path

# Resolve notebook paths from the project root, including nested notebook directories.
working_dir = Path.cwd().resolve()
project_root = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "pyproject.toml").is_file()
        and (directory / "src/feature_selector").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Open the notebook from within the feature_selector project")
os.chdir(project_root)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from feature_selector import ExperimentRun, load_config, run_experiment
from feature_selector.artifacts import load_run
from feature_selector.data import load_data
from feature_selector.evaluation import regression_metrics
from feature_selector.visualization import plot_split_predictions, plot_training_history

config = load_config(Path("configs/uzk/gru.yaml"))


## Обучение GRU

GRU прогнозирует `UZK.Q.81AY00108.FINALPOINT` на следующий шаг.
Модель обучается только на окнах с известной целью; её скейлер обучается
на этих train-целях. Validation используется для выбора весов.
Запуск сохраняет весь обученный пайплайн в отдельный каталог.


In [ ]:
run = run_experiment(config)
print(f"Results: {run.artifact_dir}")
print("GRU train windows:", sum(batch["samples"] for batch in run.model.history_[0]["batches"]))
pd.DataFrame.from_dict(run.metrics, orient="index")


## История обучения GRU

Loss GRU — MSE прогнозирования после масштабирования цели.
Показаны train loss и validation loss по эпохам; линия отмечает выбранную эпоху.
Метрики в таблицах рассчитаны в исходных единицах цели.


In [ ]:
ax = plot_training_history(
    run.model.history_, run.model.best_epoch_, title="GRU"
)
ax.set_ylabel("MSE (scaled target)")
plt.show()


## Загрузка и явная оценка test

Загружаем сохранённый пайплайн и вычисляем test без дообучения.
Этот вызов не изменяет автоматически сохранённые train/validation-отчёты.


In [ ]:
saved = load_run(run.artifact_dir)
assert isinstance(saved, ExperimentRun)
data = load_data(**config.data)
test_report = saved.predict(data, split="test")
observed = test_report.dropna(subset=["y_true"])
test_metrics = regression_metrics(observed.y_true, observed.y_pred)
reports = {**run.predictions, "test": test_report}
plot_split_predictions(reports, title="Features → GRU: train / validation / test")
plt.show()
pd.DataFrame.from_dict({**run.metrics, "test": test_metrics}, orient="index")


## Loss по тренировочным батчам

Batch loss доступен в `history_` и записывается в JSONL и TensorBoard.
Команда `uv run tensorboard --logdir runs` открывает мониторинг обучения.
Ход обучения смотрите в **Scalars**, итоговые метрики — в `metrics.json`.


In [ ]:
batch_history = pd.DataFrame([
    {"epoch": epoch["epoch"], **batch}
    for epoch in run.model.history_ for batch in epoch["batches"]
])
display(batch_history.head())
if not batch_history.empty:
    batch_history.plot(x="step", y="train_loss", title="Training batch loss")
    plt.show()


## Test-метрики по десяти сидам

Запускаем новую серию с сидами 0–9 и текущим `config`.
Предобработка обучается в первом запуске и затем остаётся фиксированной;
меняется `random_state` конечной модели. Готовые энкодеры не переобучаются.
Для ансамбля каждый сид задаёт начало последовательности сидов его GRU.

Ниже показаны test-метрики каждого запуска и среднее ± выборочное стандартное
отклонение (`ddof=1`). MAPE выражена в процентах. `n_valid` / `n_total` показывает,
для скольких запусков метрика определена. Test не используется для выбора сида.
Метрики каждого сида, сводка, модели и прогнозы сохраняются в каталоге серии.


In [ ]:
from IPython.display import display

from feature_selector.experiment import run_seed_experiments

seeds = list(range(10))
seed_results = run_seed_experiments(config, seeds=seeds)
print(f"Результаты серии: {seed_results.artifact_dir}")


In [ ]:
print("Test-метрики каждого сида:")
display(seed_results.metrics_per_seed.set_index("seed"))

print("Test-метрики: среднее ± стандартное отклонение")
seed_summary = seed_results.metrics_summary.copy()
seed_summary["mean ± std"] = [
    f"{mean:.6g} ± {std:.6g}"
    for mean, std in zip(seed_summary["mean"], seed_summary["std"], strict=True)
]
display(seed_summary.set_index("metric")[["mean ± std", "n_valid", "n_total"]])
